# 01 · SBAS InSAR กรุงเทพฯ–สมุทรปราการ–สมุทรสาคร (HyP3 + MintPy)

เปรียบเทียบอัตราการเคลื่อนตัวในแนวดิ่ง (VLM) **1 ปีก่อน** และ **1 ปีหลัง** แผ่นดินไหว 28 มี.ค. 2568

| ช่วง | วันที่ |
|---|---|
| pre  | 2024-03-28 → 2025-03-27 |
| post | 2025-03-29 → 2026-03-28 |

**ลำดับงาน**
1. ติดตั้ง + ตั้งค่า
2. ค้นภาพ Sentinel-1 เลือก track ที่คลุมพื้นที่ครบและมีภาพมากที่สุด
3. สร้างคู่ SBAS (แต่ละภาพจับคู่ภาพถัดไป `N_NEIGHBORS` ภาพ ไม่ข้ามวันแผ่นดินไหว) + คู่ co-seismic 1 คู่
4. ส่ง job ไป ASF HyP3 (ประมวลผลบน cloud ของ ASF — **รอหลายชั่วโมง ปิด Colab ได้**)
5. ดาวน์โหลด → ตัดเฉพาะพื้นที่ศึกษา (กริดเดียวกันทั้งสองช่วง) → เก็บใน Google Drive
6. รัน MintPy แยกช่วง โดยใช้ **จุดอ้างอิงเดียวกัน**
7. แปลง LOS → VLM, คำนวณ Δv และทดสอบนัยสำคัญ, แผนที่ + scatter
8. (ถ้ามีไฟล์ขอบเขต) สถิติรายเขต/อำเภอ

**ต้องมี:** บัญชี NASA Earthdata (ฟรี) — https://urs.earthdata.nasa.gov

> ทุก cell รันซ้ำได้: งานที่ทำแล้ว (job ที่ส่งแล้ว, ไฟล์ที่ตัดแล้ว) จะถูกข้าม ถ้า Colab หลุดให้รัน cell 1–2 แล้วข้ามไปขั้นที่ค้างอยู่

## 1. ติดตั้ง + เชื่อม Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q asf_search hyp3_sdk mintpy

## 2. ตั้งค่า

In [ ]:
import math, shutil, subprocess
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ที่เก็บผลถาวร (Drive) และพื้นที่ชั่วคราว (ดิสก์ Colab ~100 GB, หายเมื่อ session จบ)
WORK = Path('/content/drive/MyDrive/bkk_subsidence')
SCRATCH = Path('/content/scratch')
WORK.mkdir(parents=True, exist_ok=True)
SCRATCH.mkdir(parents=True, exist_ok=True)

# กรอบสี่เหลี่ยมคลุม กรุงเทพฯ + สมุทรปราการ + สมุทรสาคร (lon/lat)
AOI_WKT = 'POLYGON((99.95 13.40, 101.00 13.40, 101.00 14.00, 99.95 14.00, 99.95 13.40))'

EQ_DATE = date(2025, 3, 28)
PERIODS = {
    'pre':  (date(2024, 3, 28), date(2025, 3, 27)),
    'post': (date(2025, 3, 29), date(2026, 3, 28)),
}

# ใช้ Sentinel-1A อย่างเดียวเพื่อให้ทั้งสองช่วงมีรอบถ่ายภาพ 12 วันเท่ากัน
# (ช่วง post มี S1C ด้วย — ถ้าอยากใช้ เพิ่ม 'S1C' แต่ช่วง post จะถี่กว่า pre)
SATELLITES = ['S1A']

N_NEIGHBORS = 3          # จำนวนคู่ต่อภาพ (SBAS network)
LOOKS, PIXEL = '20x4', 80  # 80 ม. ('10x2', 40 ถ้าต้องการละเอียดขึ้น แต่ใช้ credit/เวลามากกว่า)
MIN_COVERAGE = 0.98      # ภาพต้องคลุม AOI อย่างน้อย 98%
TRACK = None             # None = เลือกอัตโนมัติ, หรือกำหนดเอง เช่น ('DESCENDING', 62)
PROJECT = 'bkk_subs'     # prefix ชื่อ job ใน HyP3
EPSG = 32647             # UTM 47N

## 3. ค้นภาพ Sentinel-1 และเลือก track

In [ ]:
import asf_search as asf
from shapely import wkt
from shapely.geometry import shape

aoi = wkt.loads(AOI_WKT)
results = asf.geo_search(
    platform=asf.PLATFORM.SENTINEL1,
    processingLevel=asf.PRODUCT_TYPE.SLC,
    beamMode=asf.BEAMMODE.IW,
    intersectsWith=AOI_WKT,
    start=PERIODS['pre'][0].isoformat(),
    end=PERIODS['post'][1].isoformat() + 'T23:59:59Z',
)

scenes = pd.DataFrame([{
    'scene': r.properties['sceneName'],
    'path': r.properties['pathNumber'],
    'direction': r.properties['flightDirection'],
    'time': pd.to_datetime(r.properties['startTime']),
    'coverage': shape(r.geometry).intersection(aoi).area / aoi.area,
} for r in results])
scenes['date'] = scenes.time.dt.date
scenes = scenes[scenes.scene.str[:3].isin(SATELLITES)]

def period_of(d):
    for name, (start, end) in PERIODS.items():
        if start <= d <= end:
            return name
    return None

scenes['period'] = scenes.date.map(period_of)
scenes = scenes.dropna(subset=['period'])

# วันเดียวกันอาจมีหลาย frame: เก็บภาพที่คลุม AOI มากที่สุด
scenes = (scenes.sort_values('coverage', ascending=False)
                .drop_duplicates(['direction', 'path', 'date'])
                .query('coverage >= @MIN_COVERAGE')
                .sort_values('time'))

summary = scenes.pivot_table(index=['direction', 'path'], columns='period',
                             values='scene', aggfunc='count', fill_value=0)
summary = summary.reindex(columns=list(PERIODS), fill_value=0)
print(summary)
if summary.empty:
    raise RuntimeError('ไม่มีภาพที่คลุม AOI ครบ — ลด MIN_COVERAGE หรือหดกรอบ AOI')

if TRACK is None:
    TRACK = summary.min(axis=1).idxmax()
print('\nใช้ track:', TRACK)

stack = scenes[(scenes.direction == TRACK[0]) & (scenes.path == TRACK[1])]
stack.to_csv(WORK / 'scenes.csv', index=False)
stack.groupby('period').date.agg(['count', 'min', 'max'])

## 4. สร้างคู่ SBAS + คู่ co-seismic

In [ ]:
def sbas_pairs(df, n):
    names = df.sort_values('time').scene.tolist()
    return [(names[i], names[j]) for i in range(len(names))
            for j in range(i + 1, min(i + 1 + n, len(names)))]

pairs = {p: sbas_pairs(stack[stack.period == p], N_NEIGHBORS) for p in PERIODS}

# คู่คร่อมวันแผ่นดินไหว: ภาพสุดท้ายก่อนเหตุ กับภาพแรกหลังเหตุ
last_pre = stack[stack.period == 'pre'].sort_values('time').scene.iloc[-1]
first_post = stack[stack.period == 'post'].sort_values('time').scene.iloc[0]
pairs['coseis'] = [(last_pre, first_post)]

for p, prs in pairs.items():
    print(f'{p:7s} {len(prs):4d} คู่')

# วาด network
t = stack.set_index('scene').time
fig, ax = plt.subplots(figsize=(12, 3))
for p, color in [('pre', 'tab:blue'), ('post', 'tab:orange'), ('coseis', 'tab:red')]:
    for a, b in pairs[p]:
        ax.plot([t[a], t[b]], [0, 0], color=color, alpha=.4)
ax.scatter(t, np.zeros(len(t)), c='k', s=10, zorder=3)
ax.axvline(pd.Timestamp(EQ_DATE), color='r', ls='--', label='Earthquake 2025-03-28')
ax.set_yticks([]); ax.legend(); ax.set_title('SBAS network (timeline)')
plt.show()

## 5. ส่ง job ไป ASF HyP3

ใส่ username/password ของ Earthdata — job ที่ส่งไปแล้ว (ชื่อ + คู่ภาพเดียวกัน) จะไม่ถูกส่งซ้ำ

ผลจาก HyP3 **หมดอายุใน 14 วัน** ให้ดาวน์โหลด (ขั้นที่ 6) ภายในเวลานั้น

In [ ]:
from getpass import getpass
import hyp3_sdk as sdk

hyp3 = sdk.HyP3(username=input('Earthdata username: '), password=getpass('Earthdata password: '))
print('credits คงเหลือ:', hyp3.check_credits())

In [ ]:
def job_name(period):
    return f'{PROJECT}_{period}'

for period, prs in pairs.items():
    submitted = {tuple(j.job_parameters['granules'])
                 for j in hyp3.find_jobs(name=job_name(period)) if not j.failed()}
    todo = [pr for pr in prs if pr not in submitted]
    prepared = [hyp3.prepare_insar_job(
        a, b, name=job_name(period), looks=LOOKS,
        include_dem=True, include_look_vectors=True, apply_water_mask=True,
        include_los_displacement=(period == 'coseis'),
    ) for a, b in todo]
    for i in range(0, len(prepared), 100):
        hyp3.submit_prepared_jobs(prepared[i:i + 100])
    print(f'{period}: ส่งใหม่ {len(todo)} / ส่งแล้วก่อนหน้า {len(submitted)}')

print('credits คงเหลือ:', hyp3.check_credits())

### ตรวจสถานะ (รันซ้ำได้เรื่อย ๆ — รอจนทุก job เป็น SUCCEEDED)

In [ ]:
from collections import Counter
for period in pairs:
    batch = hyp3.find_jobs(name=job_name(period))
    print(period, dict(Counter(j.status_code for j in batch)))

## 6. ดาวน์โหลด + ตัดพื้นที่

ดาวน์โหลดทีละ job ลงดิสก์ Colab → ตัดเฉพาะ AOI บนกริด UTM 47N เดียวกันทุกไฟล์ → เก็บใน Drive → ลบไฟล์เต็ม
(ไฟล์เต็ม ~ร้อย MB/คู่ แต่ไฟล์ที่ตัดแล้วเหลือไม่กี่ MB)

In [ ]:
from osgeo import gdal
from pyproj import Transformer

gdal.UseExceptions()
tr = Transformer.from_crs(4326, EPSG, always_xy=True)
lon0, lat0, lon1, lat1 = aoi.bounds
xs, ys = zip(*[tr.transform(x, y) for x in (lon0, lon1) for y in (lat0, lat1)])
BOUNDS = (math.floor(min(xs) / PIXEL) * PIXEL, math.floor(min(ys) / PIXEL) * PIXEL,
          math.ceil(max(xs) / PIXEL) * PIXEL, math.ceil(max(ys) / PIXEL) * PIXEL)

SUFFIXES = ['unw_phase', 'corr', 'los_disp']
# DEM/มุมมอง/water mask เหมือนกันทุกคู่ใน track เดียวกัน เก็บแค่ชุดเดียวต่อช่วงเพื่อประหยัดที่ใน Drive
GEOMETRY = ['dem', 'lv_theta', 'lv_phi', 'water_mask']

def clip_product(prod_dir, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    name = prod_dir.name
    shutil.copy(prod_dir / f'{name}.txt', out_dir)   # MintPy อ่าน metadata จากไฟล์นี้
    has_geometry = any(out_dir.parent.glob('*/*_dem_clip.tif'))
    for s in SUFFIXES + ([] if has_geometry else GEOMETRY):
        src = prod_dir / f'{name}_{s}.tif'
        if src.exists():
            gdal.Warp(str(out_dir / f'{name}_{s}_clip.tif'), str(src),
                      dstSRS=f'EPSG:{EPSG}', outputBounds=BOUNDS, xRes=PIXEL, yRes=PIXEL,
                      targetAlignedPixels=True, resampleAlg='near',
                      creationOptions=['COMPRESS=DEFLATE'])

for period in pairs:
    out_root = WORK / period / 'hyp3'
    jobs = [j for j in hyp3.find_jobs(name=job_name(period)) if j.succeeded() and not j.expired()]
    for k, job in enumerate(jobs, 1):
        name = Path(job.files[0]['filename']).stem
        if (out_root / name / f'{name}_unw_phase_clip.tif').exists():
            continue
        zip_path = job.download_files(SCRATCH)[0]
        prod_dir = sdk.util.extract_zipped_product(zip_path)
        clip_product(prod_dir, out_root / name)
        shutil.rmtree(prod_dir)
        print(f'{period} {k}/{len(jobs)} {name}')
    print(period, 'พร้อม:', len(list(out_root.glob('*/*_unw_phase_clip.tif'))), 'คู่')

!du -sh {WORK}

### ตรวจ co-seismic (คู่คร่อมวันแผ่นดินไหว)
กรุงเทพฯ อยู่ห่างศูนย์กลาง ~1,000 กม. คาดว่า **ไม่เห็น** การเคลื่อนตัวขณะเกิดเหตุ — ถ้าภาพเป็น noise/สีเรียบ ๆ แปลว่าแยกสองช่วงได้ถูกต้อง

In [ ]:
import rasterio

f = next((WORK / 'coseis' / 'hyp3').glob('*/*_los_disp_clip.tif'))
with rasterio.open(f) as src:
    d = src.read(1, masked=True) * 1000  # m -> mm
    ext = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
plt.figure(figsize=(8, 5))
plt.imshow(d, cmap='RdBu', vmin=-30, vmax=30, extent=ext)
plt.colorbar(label='LOS displacement (mm)')
plt.title(f.name[:40]); plt.show()

## 7. MintPy (SBAS time series)

รันบนดิสก์ Colab (เร็วกว่าและ HDF5 ทำงานกับ Drive ได้ไม่ดี) แล้วคัดลอกผลกลับไป Drive

- ช่วง **pre** ให้ MintPy เลือกจุดอ้างอิงอัตโนมัติ (pixel coherence สูง)
- ช่วง **post** ใช้ **pixel เดียวกัน** (REF_Y/REF_X) — ทำได้เพราะทั้งสองช่วงตัดบนกริดเดียวกัน
- ปิด tropospheric correction (ต้องมีบัญชี ERA5) และ deramp (การทรุดตัวของกรุงเทพฯ เป็นแอ่งกว้าง deramp อาจลบสัญญาณจริง)

> ถ้าต้องการกำหนดจุดอ้างอิงเอง (เช่นจุดที่ทราบว่าค่อนข้างคงที่) ใส่ `REF_YX = 'y,x'` ก่อนรัน แล้วรันใหม่ทั้งสองช่วง

In [ ]:
REF_YX = None   # None = auto สำหรับ pre แล้วใช้ค่าเดียวกันกับ post

KEEP = ['velocity.h5', 'timeseries.h5', 'temporalCoherence.h5', 'maskTempCoh.h5',
        'avgSpatialCoh.h5', 'inputs/geometryGeo.h5', 'pic']

def run_mintpy(period, ref_yx):
    d = WORK / period / 'hyp3'
    run_dir = SCRATCH / f'mintpy_{period}'
    run_dir.mkdir(exist_ok=True)
    cfg = run_dir / f'{PROJECT}_{period}.txt'
    cfg.write_text(f'''
mintpy.load.processor           = hyp3
mintpy.load.unwFile             = {d}/*/*_unw_phase_clip.tif
mintpy.load.corFile             = {d}/*/*_corr_clip.tif
mintpy.load.demFile             = {d}/*/*_dem_clip.tif
mintpy.load.incAngleFile        = {d}/*/*_lv_theta_clip.tif
mintpy.load.azAngleFile         = {d}/*/*_lv_phi_clip.tif
mintpy.load.waterMaskFile       = {d}/*/*_water_mask_clip.tif
mintpy.reference.yx             = {ref_yx or 'auto'}
mintpy.troposphericDelay.method = no
mintpy.deramp                   = no
mintpy.topographicResidual      = yes
mintpy.save.kmz                 = no
mintpy.compute.numWorker        = 2
''')
    subprocess.run(['smallbaselineApp.py', str(cfg), '--dir', str(run_dir)], check=True)
    out = WORK / period / 'mintpy'
    for k in KEEP:
        src, dst = run_dir / k, out / k
        dst.parent.mkdir(parents=True, exist_ok=True)
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True)
        elif src.exists():
            shutil.copy(src, dst)
    return out

from mintpy.utils import readfile

out_pre = run_mintpy('pre', REF_YX)
atr = readfile.read_attribute(str(out_pre / 'velocity.h5'))
REF_YX = f"{atr['REF_Y']},{atr['REF_X']}"
print('จุดอ้างอิง (y,x):', REF_YX)
out_post = run_mintpy('post', REF_YX)

## 8. LOS → VLM, Δv และนัยสำคัญ

- **VLM = v_LOS / cos(θ)** (สมมติว่าการเคลื่อนตัวแนวราบน้อยมาก), ค่าลบ = ทรุด, หน่วย มม./ปี
- **Δv = VLM_post − VLM_pre** (ลบ = ทรุดเร็วขึ้นหลังแผ่นดินไหว)
- มีนัยสำคัญเมื่อ **|Δv| > 2·√(SD_pre² + SD_post²)**
- ใช้เฉพาะ pixel ที่ผ่าน mask temporal coherence ของ **ทั้งสองช่วง**

In [ ]:
import rasterio

OUT = WORK / 'results'
OUT.mkdir(exist_ok=True)

def export(period, dset, name, src='velocity.h5'):
    m = WORK / period / 'mintpy'
    dst = SCRATCH / f'{name}_{period}.tif'
    subprocess.run(['save_gdal.py', str(m / src), '-d', dset, '-o', str(dst),
                    '-m', str(m / 'maskTempCoh.h5')], check=True)
    with rasterio.open(dst) as s:
        return s.read(1).astype('float64'), s.profile

def write(arr, name, profile):
    p = profile.copy(); p.update(dtype='float32', nodata=np.nan, compress='deflate')
    with rasterio.open(OUT / f'{name}.tif', 'w', **p) as dst:
        dst.write(arr.astype('float32'), 1)

R = {}
for period in PERIODS:
    v, prof = export(period, 'velocity', 'vel')
    sd, _ = export(period, 'velocityStd', 'sd')
    inc, _ = export(period, 'incidenceAngle', 'inc', src='inputs/geometryGeo.h5')
    cos = np.cos(np.deg2rad(inc))
    R[period] = {'vlm': v / cos * 1000, 'sd': sd / cos * 1000}
    write(R[period]['vlm'], f'vlm_{period}', prof)
    write(R[period]['sd'], f'sd_{period}', prof)

valid = np.isfinite(R['pre']['vlm']) & np.isfinite(R['post']['vlm'])
dv = np.where(valid, R['post']['vlm'] - R['pre']['vlm'], np.nan)
thr = 2 * np.sqrt(R['pre']['sd'] ** 2 + R['post']['sd'] ** 2)
sig = np.full(dv.shape, np.nan)
sig[valid] = 0
sig[valid & (dv < -thr)] = -1   # ทรุดเร็วขึ้นอย่างมีนัยสำคัญ
sig[valid & (dv > thr)] = 1     # ทรุดช้าลง / ยกตัวอย่างมีนัยสำคัญ
write(dv, 'dv', prof)
write(sig, 'dv_sig', prof)

n = valid.sum()
print(f'pixel ที่ใช้ได้ทั้งสองช่วง: {n:,}')
for p in PERIODS:
    v = R[p]['vlm'][valid]
    print(f'{p:5s} VLM median {np.median(v):6.1f} มม./ปี | ทรุด > 10 มม./ปี {np.mean(v < -10):5.1%}')
print(f'Δv median {np.nanmedian(dv):.1f} มม./ปี')
print(f'ทรุดเร็วขึ้นอย่างมีนัยสำคัญ {np.sum(sig == -1) / n:.1%} | ช้าลง {np.sum(sig == 1) / n:.1%} '
      f'| ไม่ต่าง {np.sum(sig == 0) / n:.1%}')

### แผนที่

In [ ]:
from matplotlib.colors import ListedColormap

b = rasterio.open(OUT / 'dv.tif').bounds
ext = [b.left, b.right, b.bottom, b.top]
fig, axs = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
for ax, p in zip(axs[0], PERIODS):
    im = ax.imshow(R[p]['vlm'], cmap='RdYlBu', vmin=-30, vmax=10, extent=ext)
    ax.set_title(f'VLM {p} (mm/yr)')
fig.colorbar(im, ax=axs[0], shrink=.8)
im = axs[1, 0].imshow(dv, cmap='RdBu', vmin=-10, vmax=10, extent=ext)
axs[1, 0].set_title('Δv = post − pre (mm/yr)'); fig.colorbar(im, ax=axs[1, 0], shrink=.8)
im = axs[1, 1].imshow(sig, cmap=ListedColormap(['#d7301f', '#dddddd', '#2b8cbe']),
                      vmin=-1.5, vmax=1.5, extent=ext)
axs[1, 1].set_title('Significant change: red = faster subsidence, blue = slower')
fig.savefig(OUT / 'maps.png', dpi=200); plt.show()

### Scatter pre vs post (ถ้าจุดเกาะเส้น 1:1 = อัตราทรุดตัวไม่เปลี่ยน)

In [ ]:
x, y = R['pre']['vlm'][valid], R['post']['vlm'][valid]
fig, ax = plt.subplots(figsize=(6, 6))
hb = ax.hexbin(x, y, gridsize=120, bins='log', cmap='viridis', extent=(-40, 15, -40, 15))
ax.plot([-40, 15], [-40, 15], 'r--', lw=1, label='1:1')
ax.set_xlabel('VLM pre (mm/yr)'); ax.set_ylabel('VLM post (mm/yr)')
ax.legend(); fig.colorbar(hb, label='log10(pixel)')
print(f'r = {np.corrcoef(x, y)[0, 1]:.2f}')
fig.savefig(OUT / 'scatter_pre_post.png', dpi=200); plt.show()

## 9. สถิติรายเขต/อำเภอ (ทางเลือก)

อัปโหลด shapefile ขอบเขตอำเภอจาก HDX (COD-AB Thailand, ระดับ adm2) ไว้ที่ `WORK/boundaries/`

In [ ]:
import geopandas as gpd
from rasterio.features import rasterize

shp = next((WORK / 'boundaries').glob('*adm2*.shp'), None)
if shp is None:
    print('ไม่พบไฟล์ขอบเขต — ข้ามขั้นนี้')
else:
    adm = gpd.read_file(shp)
    adm = adm[adm.ADM1_EN.isin(['Bangkok', 'Samut Prakan', 'Samut Sakhon'])].to_crs(EPSG).reset_index(drop=True)
    zone = rasterize(zip(adm.geometry, adm.index + 1), out_shape=dv.shape,
                     transform=prof['transform'], fill=0, dtype='int32')
    px = pd.DataFrame({'zone': zone.ravel(), 'vlm_pre': R['pre']['vlm'].ravel(),
                       'vlm_post': R['post']['vlm'].ravel(), 'dv': dv.ravel(), 'sig': sig.ravel()})
    px = px[(px.zone > 0) & px.sig.notna()]
    g = px.groupby('zone')
    table = pd.DataFrame({
        'n_pixel': g.size(),
        'vlm_pre_median': g.vlm_pre.median(),
        'vlm_post_median': g.vlm_post.median(),
        'dv_median': g.dv.median(),
        'pct_faster': g.sig.apply(lambda s: 100 * (s == -1).mean()),
        'pct_slower': g.sig.apply(lambda s: 100 * (s == 1).mean()),
    })
    table = adm[['ADM1_EN', 'ADM2_EN']].join(table.set_index(table.index - 1))
    table = table.sort_values('vlm_post_median')
    table.to_csv(OUT / 'district_stats.csv', index=False)
    display(table.round(1))